# Warsaw Real Estate: Data Scraping
This cell extracts live rental data from real estate portals, utilizing an O(1) Set for deduplication and auto-termination logic.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

seen_urls = set() 
pages_to_scrape = 76
apartments_data = []

for page_number in range(1, pages_to_scrape + 1):
    if page_number == 1:
        url = "https://www.nieruchomosci-online.pl/szukaj.html?3,mieszkanie,wynajem,,Warszawa:20571"
    else:
        url = f"https://www.nieruchomosci-online.pl/szukaj.html?3,mieszkanie,wynajem,,Warszawa:20571&p={page_number}"

    print(f"\n--- Scraping Page {page_number} ---")
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        listings = soup.select('div[id^="off-inner_"]')

        if len(listings) == 0:
            print("No more listings found. Stopping.")
            break

        new_listings_on_this_page = 0

        for listing in listings:
            title_tag = listing.find('h2', class_='name')
            inner_link = title_tag.find('a') if title_tag else None
            offer_url = inner_link['href'] if inner_link else None

            if not offer_url: continue
            if not offer_url.startswith('http'): offer_url = "https://www.nieruchomosci-online.pl" + offer_url
            if offer_url in seen_urls: continue

            seen_urls.add(offer_url)
            new_listings_on_this_page += 1

            try:
                price_container = listing.find('p', class_='primary-display')
                price_tag = price_container.find('span') if price_container else None
                price = price_tag.text.strip() if price_tag else None

                size_tag = listing.find('span', class_='area')
                size = size_tag.text.strip() if size_tag else None

                district_tag = listing.find('a', class_='margin-right4')
                district = district_tag.text.strip(' ,') if district_tag else None

                raw_title = title_tag.text.strip() if title_tag else ""
                street_name = None
                if "ul." in raw_title: street_name = raw_title.split("ul.")[1].strip()
                elif "," in raw_title: street_name = raw_title.split(",")[-1].strip()
                else: street_name = raw_title

                rooms_icon = listing.find('em', class_='icon-data-rooms')
                rooms = None
                if rooms_icon:
                    rooms_parent = rooms_icon.find_parent('div', class_='attributes__box--item')
                    rooms_strong = rooms_parent.find('strong') if rooms_parent else None
                    rooms = rooms_strong.text.strip() if rooms_strong else None

                apartments_data.append({
                    'Street': street_name, 'Raw_Title': raw_title, 'Number_of_Rooms': rooms,
                    'Price': price, 'Size_m2': size, 'District': district, 'URL': offer_url
                })
            except Exception:
                continue

        print(f"Added {new_listings_on_this_page} new unique listings.")
        if new_listings_on_this_page == 0: break
        
        time.sleep(random.uniform(2.5, 4.5))

    except Exception as e:
        print(f"Connection error: {e}")
        break

df_raw = pd.DataFrame(apartments_data)
df_raw.to_csv('warsaw_apartments_master.csv', index=False, encoding='utf-8-sig')
print(f"\nTotal unique apartments collected: {len(df_raw)}")

# Data Cleaning & Preprocessing
This cell cleans raw strings, handles missing values, and enforces strict market bounds to remove extreme outliers.

In [ ]:
import pandas as pd

df = pd.read_csv('warsaw_apartments_master.csv')

# Preprocess numeric columns
df['Price'] = df['Price'].astype(str).str.replace('zł', '').str.replace(r'\s+', '', regex=True)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df['Size_m2'] = df['Size_m2'].astype(str).str.replace('m²', '').str.replace(',', '.').str.replace(r'\s+', '', regex=True)
df['Size_m2'] = pd.to_numeric(df['Size_m2'], errors='coerce')

df['Number_of_Rooms'] = pd.to_numeric(df['Number_of_Rooms'], errors='coerce')

# Handle missing data & outliers
df = df.dropna(subset=['Price', 'Size_m2'])
df = df[(df['Price'] >= 1000) & (df['Price'] <= 30000)]
df = df[(df['Size_m2'] >= 10) & (df['Size_m2'] <= 300)]

df.to_csv('warsaw_apartments_CLEAN.csv', index=False, encoding='utf-8-sig')
print(f"Cleaned dataset shape: {df.shape[0]} records")
df.head()

# Geospatial Feature Engineering
This cell maps raw addresses to coordinates and calculates the geodesic distance to the nearest Warsaw Metro station.

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time

df = pd.read_csv('warsaw_apartments_CLEAN.csv')

metro_stations = {
    "M1 - Młociny": (52.2908, 20.9292), "M1 - Centrum": (52.2310, 21.0106), 
    "M1 - Kabaty": (52.1310, 21.0665), "M2 - Bemowo": (52.2384, 20.9125),
    "M2 - Rondo Daszyńskiego": (52.2295, 20.9840), "M2 - Bródno": (52.2928, 21.0286),
    # Note: Add the remaining 32 stations from your full dictionary here!
}

geolocator = Nominatim(user_agent="warsaw_real_estate_portfolio_project")
latitudes, longitudes, min_distances, nearest_stations = [], [], [], []

for index, row in df.iterrows():
    street = row['Street']
    if pd.isna(street) or street in ['None', '']:
        latitudes.append(None); longitudes.append(None); min_distances.append(None); nearest_stations.append(None)
        continue

    try:
        location = geolocator.geocode(f"{street}, Warszawa, Polska")
        if location:
            apt_coords = (location.latitude, location.longitude)
            latitudes.append(location.latitude); longitudes.append(location.longitude)

            shortest_dist = float('inf')
            closest_station = ""
            for name, coords in metro_stations.items():
                dist = geodesic(apt_coords, coords).kilometers
                if dist < shortest_dist:
                    shortest_dist = dist
                    closest_station = name
            
            min_distances.append(round(shortest_dist, 2))
            nearest_stations.append(closest_station)
        else:
            latitudes.append(None); longitudes.append(None); min_distances.append(None); nearest_stations.append(None)
    except:
        latitudes.append(None); longitudes.append(None); min_distances.append(None); nearest_stations.append(None)
    
    time.sleep(1.1) # Respect API rate limits

df['Latitude'] = latitudes
df['Longitude'] = longitudes
df['Dist_to_Metro_km'] = min_distances
df['Nearest_Metro'] = nearest_stations

df.to_csv('warsaw_apartments_FINAL.csv', index=False, encoding='utf-8-sig')
print("Geospatial features mapped and saved.")

# Machine Learning Pipeline
This cell reduces categorical cardinality, applies one-hot encoding, and trains a Random Forest Regressor to predict prices.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

df = pd.read_csv('warsaw_apartments_FINAL.csv')

# Comprehensive dimensionality reduction map
district_map = {
    'Centrum': 'Śródmieście', 'Służewiec': 'Mokotów', 'Kabaty': 'Ursynów', 'Gocław': 'Praga-Południe',
    'Nowolipki': 'Śródmieście', 'Skorosze': 'Ursus', 'Zacisze': 'Targówek', 'Powązki': 'Wola',
    # Note: Ensure your full 130+ neighborhood dictionary is pasted here to catch any scraped anomalies!
}

df['District_Clean'] = df['District'].map(district_map).fillna(df['District'])

official_18_districts = [
    'Bemowo', 'Białołęka', 'Bielany', 'Mokotów', 'Ochota', 'Praga-Południe',
    'Praga-Północ', 'Rembertów', 'Śródmieście', 'Targówek', 'Ursus', 'Ursynów',
    'Wawer', 'Wesoła', 'Wilanów', 'Włochy', 'Wola', 'Żoliborz'
]

df = df[df['District_Clean'].isin(official_18_districts)]
df = df.dropna(subset=['Price', 'Size_m2', 'Dist_to_Metro_km'])
df = df[(df['Price'] >= 1600) & (df['Price'] <= 14000)]
df = df[(df['Size_m2'] >= 18) & (df['Size_m2'] <= 150)]

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['District_Clean'], prefix='Distr')
features = ['Size_m2', 'Dist_to_Metro_km'] + [col for col in df_encoded.columns if 'Distr_' in col]
X = df_encoded[features]
y = df_encoded['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Random Forest Regressor on {len(df)} records...")
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.0f} PLN")

joblib.dump(rf_model, 'warsaw_rent_model.pkl')
joblib.dump(features, 'model_features.pkl')
print("Model saved for production.")